In [1]:
!pip install -q transformers datasets accelerate einops

# Mamba bản ổn định KHÔNG cần build causal-conv1d
!pip install -q mamba-ssm --no-build-isolation

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 94.0 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which 

In [2]:
import torch
import time
import numpy as np

from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, BertForSequenceClassification

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

DEVICE: cuda


In [3]:
dataset = load_dataset("ag_news")

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_fn(example, max_len):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=max_len
    )

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
def get_loader(max_len, batch_size=8, n_samples=512):
    ds = dataset["train"].select(range(n_samples))
    ds = ds.map(lambda x: tokenize_fn(x, max_len), batched=True)

    ds.set_format(type="torch", columns=["input_ids", "label"])

    return DataLoader(ds, batch_size=batch_size)

In [5]:
# ===== BERT =====
def get_bert():
    model = BertForSequenceClassification.from_pretrained(
        "bert-base-uncased",
        num_labels=4
    )
    return model.to(DEVICE)

# ===== MAMBA =====
from mamba_ssm import Mamba

class MambaClassifier(torch.nn.Module):
    def __init__(self, vocab_size, d_model=768):
        super().__init__()

        self.embedding = torch.nn.Embedding(vocab_size, d_model)
        self.mamba = Mamba(d_model=d_model)
        self.fc = torch.nn.Linear(d_model, 4)

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        x = self.mamba(x)
        x = x.mean(dim=1)
        return self.fc(x)

In [6]:
def train_quick(model, loader, steps=200):
    model.train()

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    criterion = torch.nn.CrossEntropyLoss()

    step = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        optimizer.zero_grad()

        outputs = model(input_ids)

        if isinstance(outputs, torch.Tensor):
            logits = outputs
        else:
            logits = outputs.logits

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        step += 1
        if step >= steps:
            break

In [7]:
from sklearn.metrics import accuracy_score

def evaluate(model, loader):
    model.eval()

    preds = []
    labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            y = batch["label"].to(DEVICE)

            outputs = model(input_ids)

            if isinstance(outputs, torch.Tensor):
                logits = outputs
            else:
                logits = outputs.logits

            pred = torch.argmax(logits, dim=1)

            preds.extend(pred.cpu().numpy())
            labels.extend(y.cpu().numpy())

    return accuracy_score(labels, preds)

In [8]:
def benchmark(model, loader):
    model.eval()

    torch.cuda.reset_peak_memory_stats()
    start = time.time()
    total_tokens = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)

            outputs = model(input_ids)

            if isinstance(outputs, torch.Tensor):
                logits = outputs
            else:
                logits = outputs.logits

            total_tokens += input_ids.numel()

    end = time.time()

    elapsed = end - start
    mem = torch.cuda.max_memory_allocated() / 1024**2

    return elapsed, total_tokens / elapsed, mem

In [9]:
seq_lengths = [128, 256, 512]
results = []

for seq_len in seq_lengths:
    print(f"\n===== {seq_len} TOKENS =====")

    train_loader = get_loader(seq_len, n_samples=1000)
    test_loader  = get_loader(seq_len, n_samples=500)

    # ===== BERT =====
    bert = get_bert()

    train_quick(bert, train_loader)
    acc_bert = evaluate(bert, test_loader)

    t_bert, th_bert, mem_bert = benchmark(bert, test_loader)

    print(f"BERT  | acc: {acc_bert:.3f} | time: {t_bert:.2f}s | {th_bert:.0f} tok/s | {mem_bert:.0f}MB")

    # ===== MAMBA =====
    mamba = MambaClassifier(vocab_size=tokenizer.vocab_size).to(DEVICE)

    train_quick(mamba, train_loader)
    acc_mamba = evaluate(mamba, test_loader)

    t_mamba, th_mamba, mem_mamba = benchmark(mamba, test_loader)

    print(f"Mamba | acc: {acc_mamba:.3f} | time: {t_mamba:.2f}s | {th_mamba:.0f} tok/s | {mem_mamba:.0f}MB")

    results.append((seq_len, acc_bert, acc_mamba, t_bert, t_mamba))


===== 128 TOKENS =====


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT  | acc: 0.740 | time: 15.05s | 17011 tok/s | 1217MB
Mamba | acc: 0.740 | time: 0.81s | 315564 tok/s | 1245MB


In [10]:
print("\n===== FINAL TABLE =====")
print("Seq | BERT_acc | Mamba_acc | BERT_time | Mamba_time")

for r in results:
    print(f"{r[0]:<4} | {r[1]:<8.3f} | {r[2]:<10.3f} | {r[3]:<10.2f} | {r[4]:<10.2f}")


===== FINAL TABLE =====
Seq | BERT_acc | Mamba_acc | BERT_time | Mamba_time
128  | 0.016    | 0.740      | 3.30       | 0.18      
256  | 0.018    | 0.738      | 8.19       | 0.40      
512  | 0.740    | 0.740      | 15.05      | 0.81      
